# Vision-Language-Action (VLA) Models | Embodied / Physical

In [1]:
# VLA Pipeline: Warehouse Robot Sorting Items Into Bins
import math
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

In [2]:
@dataclass
class Item:
    label: str
    color: str
    position: Tuple[float, float]   # (x, y) on conveyor
    target_bin: str                  # which bin it belongs in

@dataclass
class PickPlaceAction:
    item_label: str
    pick_pos: Tuple[float, float]
    place_bin: str
    distance: float

@dataclass
class WorldState:
    robot_pos: Tuple[float, float] = (0.0, 0.0)
    gripper_holding: str = ""
    bins: Dict[str, List[str]] = field(default_factory=lambda: {"A": [], "B": [], "C": []})

def dist(a: Tuple[float, float], b: Tuple[float, float]) -> float:
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)

# Bin locations in the warehouse
BIN_POSITIONS = {"A": (5.0, 0.0), "B": (5.0, 3.0), "C": (5.0, 6.0)}

In [3]:
# --- PERCEIVE: filter scene for items needing sorting ---
def perceive(scene: List[Item], already_sorted: Dict[str, List[str]]) -> List[Item]:
    """Identify unsorted items by checking what's NOT already in the correct bin."""
    unsorted = []
    for item in scene:
        if item.label not in already_sorted.get(item.target_bin, []):
            unsorted.append(item)
    print(f"[Perceive] {len(unsorted)}/{len(scene)} items need sorting")
    return unsorted

# --- PLAN: generate optimized pick-place sequence ---
def plan(items: List[Item], robot_pos: Tuple[float, float]) -> List[PickPlaceAction]:
    """Sort actions by distance from current robot position (greedy nearest-first)."""
    remaining = list(items)
    actions, pos = [], robot_pos
    while remaining:
        remaining.sort(key=lambda it: dist(pos, it.position))
        item = remaining.pop(0)
        d = dist(pos, item.position) + dist(item.position, BIN_POSITIONS[item.target_bin])
        actions.append(PickPlaceAction(item.label, item.position, item.target_bin, round(d, 2)))
        pos = BIN_POSITIONS[item.target_bin]  # robot ends at the bin after drop-off
    print(f"[Plan] {len(actions)} actions, total distance: {sum(a.distance for a in actions):.1f}")
    return actions

# --- ACT: execute actions, updating world state ---
def act(actions: List[PickPlaceAction], state: WorldState) -> WorldState:
    for a in actions:
        state.robot_pos = a.pick_pos
        state.gripper_holding = a.item_label
        print(f"[Act] Pick '{a.item_label}' at {a.pick_pos}")
        state.robot_pos = BIN_POSITIONS[a.place_bin]
        state.bins[a.place_bin].append(a.item_label)
        state.gripper_holding = ""
        print(f"[Act] Place '{a.item_label}' in bin {a.place_bin}")
    return state

# --- VERIFY: check all items reached correct bins ---
def verify(scene: List[Item], state: WorldState) -> bool:
    for item in scene:
        if item.label not in state.bins.get(item.target_bin, []):
            print(f"[Verify] FAIL: '{item.label}' not in bin {item.target_bin}")
            return False
    print(f"[Verify] SUCCESS: all {len(scene)} items in correct bins")
    return True

In [4]:
# --- Run the full VLA pipeline ---
scene = [
    Item("bolt_17",   "silver", (1.0, 4.5), "C"),
    Item("widget_03", "red",    (2.0, 1.0), "A"),
    Item("gear_11",   "black",  (1.5, 2.5), "B"),
    Item("widget_08", "red",    (3.0, 0.5), "A"),
    Item("gear_22",   "black",  (0.5, 3.0), "B"),
]

state = WorldState()
unsorted = perceive(scene, state.bins)
actions  = plan(unsorted, state.robot_pos)
state    = act(actions, state)

# --- VERIFY with explicit assertions ---
assert verify(scene, state), "Sorting verification failed!"
for item in scene:
    assert item.label in state.bins[item.target_bin], f"{item.label} missing from bin {item.target_bin}"

# Visual representation of bins
print("\n--- Bin Contents ---")
for bin_name in sorted(state.bins):
    items = state.bins[bin_name]
    print(f"  Bin {bin_name}: [{', '.join(items)}]" if items else f"  Bin {bin_name}: [empty]")

# --- FAILURE CASE: item can't reach its bin (blocked path) ---
print("\n--- Failure Scenario: Blocked Path ---")
BLOCKED_BINS = {"C"}  # Simulate bin C is physically blocked
blocked_item = Item("sensor_99", "blue", (2.0, 5.0), "C")
scene_with_blocked = scene + [blocked_item]

state2 = WorldState()
unsorted2 = perceive(scene_with_blocked, state2.bins)
actions2 = plan(unsorted2, state2.robot_pos)
# Act but skip items whose target bin is blocked
for a in actions2:
    if a.place_bin in BLOCKED_BINS:
        print(f"[Act] BLOCKED: Cannot place '{a.item_label}' in bin {a.place_bin} — path obstructed!")
    else:
        state2.robot_pos = a.pick_pos
        state2.bins[a.place_bin].append(a.item_label)
        state2.robot_pos = BIN_POSITIONS[a.place_bin]

ok = verify(scene_with_blocked, state2)
print(f"Partial completion: {sum(len(v) for v in state2.bins.values())}/{len(scene_with_blocked)} items sorted")

[Perceive] 5/5 items need sorting
[Plan] 5 actions, total distance: 35.8
[Act] Pick 'widget_03' at (2.0, 1.0)
[Act] Place 'widget_03' in bin A
[Act] Pick 'widget_08' at (3.0, 0.5)
[Act] Place 'widget_08' in bin A
[Act] Pick 'gear_11' at (1.5, 2.5)
[Act] Place 'gear_11' in bin B
[Act] Pick 'bolt_17' at (1.0, 4.5)
[Act] Place 'bolt_17' in bin C
[Act] Pick 'gear_22' at (0.5, 3.0)
[Act] Place 'gear_22' in bin B
[Verify] SUCCESS: all 5 items in correct bins

--- Bin Contents ---
  Bin A: [widget_03, widget_08]
  Bin B: [gear_11, gear_22]
  Bin C: [bolt_17]

--- Failure Scenario: Blocked Path ---
[Perceive] 6/6 items need sorting
[Plan] 6 actions, total distance: 42.6
[Act] BLOCKED: Cannot place 'sensor_99' in bin C — path obstructed!
[Act] BLOCKED: Cannot place 'bolt_17' in bin C — path obstructed!
[Verify] FAIL: 'bolt_17' not in bin C
Partial completion: 4/6 items sorted
